In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import pybedtools

import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    balanced_accuracy_score,
    confusion_matrix,
    matthews_corrcoef,
    multilabel_confusion_matrix,
    precision_recall_fscore_support,
)
from tqdm import tqdm
import multiprocessing as mp
import os


In [ ]:
from utils.tcga_segmentation_workflow import (
    OUT_DIR,
    TRAIN_SAMPLE_ID,
    load_tcga_samples_info,
    select_tcga_samples,
    write_selected_samples_manifest,
    load_tcga_sample_beta_dataframe,
    run_segmentation_for_sample,
    load_meth_ref,
)


In [ ]:
all_samples = load_tcga_samples_info()
all_samples.sample_type.value_counts()

In [ ]:
AUTOSOMES = [f"chr{i}" for i in range(1, 23)]
classification_mode = "tumor_types_plus_normal"
tumor_types = None
cv_n_splits = 5
top_n_pmd_regions = 2
max_preload_workers = 70
parallelize_folds = True
max_fold_workers = min(2, cv_n_splits)
max_feature_workers = min(4, os.cpu_count() or 1)
MAX_FEATURES = None
random_region_seed = 42

base_dir = Path("/uufs/chpc.utah.edu/common/home/u0914269/clement/projects/20260624_methylseg/results/05_tcga_classification_analysis/segmentation")
PMD_SUMMARY_NAME = "segments_PMD.bed"
genome_file = "/uufs/chpc.utah.edu/common/home/u0914269/clement/projects/20260624_methylseg/analysis/07_hmms/00_data_prep/data/hg38.genome"



In [ ]:
def summarize_tumor_projects(all_samples, cv_n_splits):
    eligible = all_samples.copy()
    eligible = eligible[eligible["sample_id"].astype(str).str.startswith("TCGA-")].copy()
    eligible = eligible[eligible["project_id"].astype(str).str.startswith("TCGA-")].copy()
    eligible = eligible[eligible["methylation_file"].notna()].copy()
    eligible = eligible[eligible["sample_type"].isin(["Primary Tumor", "Solid Tissue Normal"])].copy()

    summary = eligible.groupby(["project_id", "sample_type"]).size().unstack(fill_value=0)
    for col in ["Primary Tumor", "Solid Tissue Normal"]:
        if col not in summary.columns:
            summary[col] = 0

    summary = summary[["Primary Tumor", "Solid Tissue Normal"]].reset_index()
    summary = summary.rename(columns={
        "Primary Tumor": "n_tumors",
        "Solid Tissue Normal": "n_normals",
    })
    summary["has_matched_normal"] = (summary["n_tumors"] > 0) & (summary["n_normals"] > 0)
    summary["tumor_cv_feasible"] = summary["n_tumors"] >= cv_n_splits
    summary["tumor_plus_normal_eligible"] = summary["has_matched_normal"] & summary["tumor_cv_feasible"]
    return summary.sort_values(["tumor_plus_normal_eligible", "n_tumors", "project_id"], ascending=[False, False, True]).reset_index(drop=True)


def get_default_tumor_types(project_summary_df, classification_mode):
    if classification_mode == "tumor_types":
        mask = project_summary_df["tumor_cv_feasible"]
    elif classification_mode == "tumor_types_plus_normal":
        mask = project_summary_df["tumor_plus_normal_eligible"]
    else:
        raise ValueError(f"Unsupported classification_mode: {classification_mode}")
    return project_summary_df.loc[mask, "project_id"].astype(str).tolist()


def select_multiclass_cohort(all_samples, project_summary_df, classification_mode, tumor_types=None):
    eligible = all_samples.copy()
    eligible = eligible[eligible["sample_id"].astype(str).str.startswith("TCGA-")].copy()
    eligible = eligible[eligible["project_id"].astype(str).str.startswith("TCGA-")].copy()
    eligible = eligible[eligible["methylation_file"].notna()].copy()
    eligible = eligible[eligible["sample_type"].isin(["Primary Tumor", "Solid Tissue Normal"])].copy()

    selected_tumor_types = get_default_tumor_types(project_summary_df, classification_mode) if tumor_types is None else [str(x) for x in tumor_types]
    if not selected_tumor_types:
        raise ValueError("No tumor types were selected for this analysis.")

    unknown = sorted(set(selected_tumor_types).difference(project_summary_df["project_id"].astype(str)))
    if unknown:
        raise ValueError(f"Unknown tumor types requested: {unknown}")

    if classification_mode == "tumor_types_plus_normal":
        unmatched = project_summary_df.loc[
            project_summary_df["project_id"].astype(str).isin(selected_tumor_types) & ~project_summary_df["has_matched_normal"],
            "project_id",
        ].astype(str).tolist()
        if unmatched:
            raise ValueError(
                "tumor_types_plus_normal requires matched-normal tumor types only. "
                f"These project IDs do not have matched normals: {unmatched}"
            )
        cohort = eligible[
            eligible["project_id"].astype(str).isin(selected_tumor_types)
            & eligible["sample_type"].isin(["Primary Tumor", "Solid Tissue Normal"])
        ].copy()
    elif classification_mode == "tumor_types":
        cohort = eligible[
            eligible["project_id"].astype(str).isin(selected_tumor_types)
            & (eligible["sample_type"] == "Primary Tumor")
        ].copy()
    else:
        raise ValueError(f"Unsupported classification_mode: {classification_mode}")

    cohort = cohort.drop_duplicates(subset=["sample_id"]).reset_index(drop=True)
    if cohort.empty:
        raise ValueError("The selected cohort is empty after applying the multiclass filters.")
    return cohort, selected_tumor_types


def build_multiclass_labels(samples_df, classification_mode):
    sample_meta = samples_df.set_index("sample_id")
    if classification_mode == "tumor_types":
        return sample_meta["project_id"].astype(str)
    if classification_mode == "tumor_types_plus_normal":
        return pd.Series(
            np.where(
                sample_meta["sample_type"].astype(str) == "Solid Tissue Normal",
                "Normal",
                sample_meta["project_id"].astype(str),
            ),
            index=sample_meta.index,
            name="label",
        )
    raise ValueError(f"Unsupported classification_mode: {classification_mode}")


def validate_multiclass_cohort(analysis_samples_df, classification_mode, cv_n_splits):
    labels = build_multiclass_labels(analysis_samples_df, classification_mode)
    label_counts = labels.value_counts().rename_axis("label").reset_index(name="n_samples")
    too_small = label_counts.loc[label_counts["n_samples"] < cv_n_splits]
    if not too_small.empty:
        raise ValueError(
            f"Each class needs at least {cv_n_splits} samples for StratifiedKFold. "
            f"Too-small classes: {too_small.to_dict(orient='records')}"
        )
    if label_counts.shape[0] < 2:
        raise ValueError("Multiclass analysis requires at least 2 distinct classes.")
    return label_counts


def build_analysis_label(classification_mode, selected_tumor_types):
    base_label = {
        "tumor_types": "Tumor-type multiclass classification",
        "tumor_types_plus_normal": "Tumor types plus Normal multiclass classification",
    }[classification_mode]
    return f"{base_label} ({len(selected_tumor_types)} tumor types)"


### Analysis mode

This notebook supports **single-run multiclass classification** with two modes: `classification_mode = "tumor_types"` for tumor-only cancer-type prediction, and `classification_mode = "tumor_types_plus_normal"` for one class per tumor type plus a shared `Normal` class.

All PMD discovery, random-region generation, feature construction, and imputation are performed **inside each training fold only**. PMDs are selected as the top `N` regions **per cancer type** within the training fold, then overlapping nominations are merged into one combined PMD feature set before feature extraction. The random baseline remains strictly PMD-free and is generated from the non-PMD complement within each fold.

Fold parallelism is optional and can speed up CV, but it increases memory use because each worker needs access to the preloaded methylation matrix and fold-local PMD/random-region state.


In [ ]:
project_summary_df = summarize_tumor_projects(all_samples, cv_n_splits=cv_n_splits)
analysis_samples_df, selected_tumor_types = select_multiclass_cohort(
    all_samples=all_samples,
    project_summary_df=project_summary_df,
    classification_mode=classification_mode,
    tumor_types=tumor_types,
)
label_counts_df = validate_multiclass_cohort(analysis_samples_df, classification_mode, cv_n_splits)
analysis_label = build_analysis_label(classification_mode, selected_tumor_types)
preload_samples_df = analysis_samples_df.copy()
selected_cohort_summary_df = (
    analysis_samples_df.groupby(["project_id", "sample_type"]).size().rename("n_samples").reset_index().sort_values(["project_id", "sample_type"])
)
selected_project_summary_df = (
    project_summary_df.loc[project_summary_df["project_id"].astype(str).isin(selected_tumor_types), ["project_id", "n_tumors", "n_normals", "has_matched_normal", "tumor_cv_feasible", "tumor_plus_normal_eligible"]]
    .sort_values("project_id")
    .reset_index(drop=True)
)


In [ ]:
project_summary_df[["project_id", "n_tumors", "n_normals", "has_matched_normal", "tumor_cv_feasible", "tumor_plus_normal_eligible"]].head(30), selected_project_summary_df.head(30), selected_cohort_summary_df.head(30), label_counts_df


In [ ]:
meth_ref = load_meth_ref()
meth_ref.head()

In [ ]:
probe_df = meth_ref[["CpG_chrm", "CpG_beg", "CpG_end", "key"]].copy()
probe_df = probe_df.dropna(subset=["CpG_chrm", "CpG_beg", "CpG_end"])
probe_df["CpG_beg"] = probe_df["CpG_beg"].astype(int)
probe_df["CpG_end"] = probe_df["CpG_end"].astype(int)
probe_df = probe_df[probe_df["CpG_end"] > probe_df["CpG_beg"]]
probe_df = probe_df.rename(columns={
    "CpG_chrm": "chrom",
    "CpG_beg": "start",
    "CpG_end": "end",
    "key": "name",
})
probe_df = probe_df[probe_df["chrom"].isin(AUTOSOMES)].copy()
probes_bed = pybedtools.BedTool.from_dataframe(probe_df[["chrom", "start", "end", "name"]])
probe_df.head()

In [ ]:
def _load_one_sample(sample_id, meth_file):
    meth_data = load_tcga_sample_beta_dataframe(sample_id, meth_file, drop_nas=False)
    return sample_id, meth_data


def _load_one_sample_star(args):
    return _load_one_sample(*args)


def preload_all_samples_methylation_data(samples_df, meth_ref, max_workers=None):
    coord_cols = ["CpG_chrm", "CpG_beg", "CpG_end"]
    coord_df = meth_ref.loc[:, ["key", *coord_cols]].copy()
    coord_df["CpG_chrm"] = coord_df["CpG_chrm"].astype(str)
    if not coord_df["CpG_chrm"].str.startswith("chr").all():
        coord_df["CpG_chrm"] = "chr" + coord_df["CpG_chrm"].str.replace("^chr", "", regex=True)
    coord_df["CpG_beg"] = pd.to_numeric(coord_df["CpG_beg"], errors="coerce")
    coord_df["CpG_end"] = pd.to_numeric(coord_df["CpG_end"], errors="coerce")
    coord_df = coord_df.set_index("key")
    sample_beta_by_id = {}

    if max_workers is None:
        max_workers = min(len(samples_df), os.cpu_count() or 1)

    sample_jobs = [
        (str(sample_row.sample_id), sample_row.methylation_file)
        for sample_row in samples_df.itertuples(index=False)
    ]

    ctx = mp.get_context("fork") if os.name != "nt" else mp.get_context()
    with ctx.Pool(processes=max_workers) as pool:
        with tqdm(total=len(sample_jobs), desc="Loading methylation data") as pbar:
            for returned_sample_id, meth_data in pool.imap_unordered(_load_one_sample_star, sample_jobs):
                sample_beta = meth_data.set_index("probe")["beta"]
                sample_beta_by_id[returned_sample_id] = sample_beta.reindex(coord_df.index)
                pbar.update(1)

    sample_beta_df = pd.DataFrame(sample_beta_by_id)
    meth_data_df = pd.concat([coord_df, sample_beta_df], axis=1).reset_index()
    return meth_data_df


def ensure_meth_data_df_has_samples(existing_meth_data_df, samples_df, meth_ref, max_workers=None):
    required_sample_ids = samples_df["sample_id"].astype(str).tolist()
    existing_sample_ids = {str(col) for col in existing_meth_data_df.columns if str(col) not in {"key", "CpG_chrm", "CpG_beg", "CpG_end"}}
    missing_sample_ids = [sample_id for sample_id in required_sample_ids if sample_id not in existing_sample_ids]
    if not missing_sample_ids:
        return existing_meth_data_df

    print(f"Loading {len(missing_sample_ids)} missing sample columns into meth_data_df")
    missing_samples_df = samples_df.loc[samples_df["sample_id"].astype(str).isin(missing_sample_ids)].copy()
    missing_meth_data_df = preload_all_samples_methylation_data(missing_samples_df, meth_ref, max_workers=max_workers)

    existing_indexed = existing_meth_data_df.set_index("key")
    missing_indexed = missing_meth_data_df.set_index("key")
    missing_value_cols = [
        col for col in missing_indexed.columns
        if col not in {"CpG_chrm", "CpG_beg", "CpG_end"}
    ]
    combined = pd.concat([existing_indexed, missing_indexed[missing_value_cols]], axis=1)
    return combined.reset_index()


try:
    meth_data_df
except NameError:
    meth_data_df = preload_all_samples_methylation_data(preload_samples_df, meth_ref, max_workers=max_preload_workers)
else:
    meth_data_df = ensure_meth_data_df_has_samples(meth_data_df, preload_samples_df, meth_ref, max_workers=max_preload_workers)


In [ ]:
meth_data_df

In [ ]:
METRIC_COLS = [
    "mcc",
    "balanced_accuracy",
    "macro_f1",
    "macro_specificity",
    "macro_negative_precision",
]


def check_for_missing(meth_data_df, chrom, start, end, min_cpgs=1):
    subset = meth_data_df[
        (meth_data_df["CpG_chrm"].astype(str) == str(chrom))
        & (meth_data_df["CpG_beg"] >= start)
        & (meth_data_df["CpG_end"] <= end)
    ]

    if subset.empty:
        return True

    beta_cols = [
        c for c in meth_data_df.columns if c not in {"CpG_chrm", "CpG_beg", "CpG_end", "key"}
    ]
    if subset[beta_cols].isna().all(axis=0).any():
        return True
    if len(subset) < min_cpgs:
        return True
    return False


def subtract_used_span(df, chrom, used_start, used_end):
    kept = []
    for row in df.itertuples(index=False):
        if row.chr != chrom or row.end <= used_start or row.start >= used_end:
            kept.append({"chr": row.chr, "start": int(row.start), "end": int(row.end)})
            continue
        if row.start < used_start:
            kept.append({"chr": row.chr, "start": int(row.start), "end": int(used_start)})
        if row.end > used_end:
            kept.append({"chr": row.chr, "start": int(used_end), "end": int(row.end)})
    out = pd.DataFrame(kept)
    if not out.empty:
        out = out[out["end"] > out["start"]].reset_index(drop=True)
    return out


def prepare_random_region_pool(non_pmd_regions_df):
    pool = non_pmd_regions_df.copy().reset_index(drop=True)
    pool["chr"] = pool["chr"].astype(str)
    pool["length"] = pool["end"] - pool["start"]
    return pool[pool["chr"].isin(AUTOSOMES)].reset_index(drop=True)


def _sample_interval_from_row(row, interval_len, meth_data_df, min_cpgs, rng, max_tries):
    chrom = str(row["chr"])
    region_len = int(row["length"])
    interval_len = int(interval_len)
    max_offset = region_len - interval_len
    if max_offset < 0:
        return None

    for _ in range(max_tries):
        offset = 0 if max_offset == 0 else rng.randint(0, max_offset)
        used_start = int(row["start"] + offset)
        used_end = used_start + interval_len
        if not check_for_missing(meth_data_df, chrom, used_start, used_end, min_cpgs=min_cpgs):
            return {
                "chr": chrom,
                "start": used_start,
                "end": used_end,
                "requested_length": int(interval_len),
                "realized_length": int(interval_len),
                "pool_row_length": region_len,
            }
    return None


def sample_non_pmd_interval(pool, meth_data_df, requested_len, min_cpgs, rng, max_tries=200):
    if pool.empty:
        return None

    candidate_idx = list(pool.index)
    rng.shuffle(candidate_idx)

    exact_or_longer = [idx for idx in candidate_idx if int(pool.loc[idx, "length"]) >= requested_len]
    for idx in exact_or_longer:
        row = pool.loc[idx]
        chosen = _sample_interval_from_row(row, requested_len, meth_data_df, min_cpgs, rng, max_tries)
        if chosen is not None:
            chosen["match_type"] = "exact_length"
            chosen["pool_row_index"] = int(idx)
            return chosen

    shorter_idx = [idx for idx in candidate_idx if 0 < int(pool.loc[idx, "length"]) < requested_len]
    shorter_idx = sorted(shorter_idx, key=lambda idx: int(pool.loc[idx, "length"]), reverse=True)
    for idx in shorter_idx:
        row = pool.loc[idx]
        shorter_len = int(row["length"])
        chosen = _sample_interval_from_row(row, shorter_len, meth_data_df, min_cpgs, rng, max_tries)
        if chosen is not None:
            chosen["match_type"] = "shortened_interval"
            chosen["pool_row_index"] = int(idx)
            return chosen
    return None


def get_random_regions(pmd_regions_df, non_pmd_regions_df, meth_data_df, seed=42, min_cpgs=1):
    rng = random.Random(seed)
    random_regions = []
    skipped = []
    pool = prepare_random_region_pool(non_pmd_regions_df)

    for pmd_row in pmd_regions_df.itertuples(index=False):
        requested_len = int(pmd_row.end - pmd_row.start)
        chosen = sample_non_pmd_interval(pool, meth_data_df, requested_len, min_cpgs, rng)
        if chosen is None:
            skipped.append({
                "chr": pmd_row.chr,
                "start": int(pmd_row.start),
                "end": int(pmd_row.end),
                "requested_length": requested_len,
                "seed": seed,
                "reason": "no PMD-free interval with enough CpG coverage could be placed",
            })
            continue

        pool = subtract_used_span(pool[["chr", "start", "end"]], chosen["chr"], chosen["start"], chosen["end"])
        if not pool.empty:
            pool = prepare_random_region_pool(pool)

        random_regions.append({
            "chr": chosen["chr"],
            "start": chosen["start"],
            "end": chosen["end"],
            "match_type": chosen["match_type"],
            "requested_length": requested_len,
            "realized_length": chosen["realized_length"],
            "length_ratio": chosen["realized_length"] / requested_len,
            "seed": seed,
        })

    return pd.DataFrame(random_regions), pd.DataFrame(skipped), pool


def validate_random_regions(random_regions_df, non_pmd_regions_df, pmd_union_bed, genome_file):
    required_cols = {"chr", "start", "end", "requested_length", "realized_length", "match_type", "seed"}
    if random_regions_df.empty:
        return pd.DataFrame(columns=["check", "passed", "detail"])

    missing_cols = required_cols.difference(random_regions_df.columns)
    if missing_cols:
        raise ValueError(f"random_regions_df is missing required columns: {sorted(missing_cols)}")

    autosomes_ok = random_regions_df["chr"].isin(AUTOSOMES).all()
    lengths_ok = (random_regions_df["realized_length"] <= random_regions_df["requested_length"]).all()
    containment_ok = random_regions_df.apply(
        lambda row: ((non_pmd_regions_df["chr"] == row["chr"]) & (non_pmd_regions_df["start"] <= row["start"]) & (non_pmd_regions_df["end"] >= row["end"])).any(),
        axis=1,
    ).all()

    random_bed = pybedtools.BedTool.from_dataframe(random_regions_df[["chr", "start", "end"]]).sort(g=genome_file)
    overlap_df = random_bed.intersect(pmd_union_bed, u=True).to_dataframe(names=["chr", "start", "end"])
    overlap_ok = overlap_df.empty

    checks = pd.DataFrame([
        {"check": "autosomes_only", "passed": bool(autosomes_ok), "detail": int((~random_regions_df["chr"].isin(AUTOSOMES)).sum())},
        {"check": "contained_within_single_non_pmd_interval", "passed": bool(containment_ok), "detail": int(len(random_regions_df) - random_regions_df.apply(lambda row: ((non_pmd_regions_df["chr"] == row["chr"]) & (non_pmd_regions_df["start"] <= row["start"]) & (non_pmd_regions_df["end"] >= row["end"])).any(), axis=1).sum())},
        {"check": "no_pmd_overlap", "passed": bool(overlap_ok), "detail": int(len(overlap_df))},
        {"check": "realized_length_lte_requested_length", "passed": bool(lengths_ok), "detail": int((random_regions_df["realized_length"] > random_regions_df["requested_length"]).sum())},
    ])

    failed = checks[~checks["passed"]]
    if not failed.empty:
        raise ValueError(f"Random-region validation failed: {failed.to_dict(orient='records')}")

    return checks


def get_pmd_summary_path(sample_id, base_dir, summary_name=PMD_SUMMARY_NAME):
    return Path(base_dir) / "methylseg" / str(sample_id) / "out" / "hm450k" / "summary_files" / summary_name


def load_pmd_regions_for_samples(sample_ids, base_dir, summary_name=PMD_SUMMARY_NAME):
    all_pmds = []
    missing_paths = []
    for sample_id in sample_ids:
        summary_file = get_pmd_summary_path(sample_id, base_dir=base_dir, summary_name=summary_name)
        if not summary_file.exists():
            missing_paths.append(str(summary_file))
            continue
        try:
            pmd_df = pd.read_csv(summary_file, sep="	", names=["chr", "start", "end", "RegionType"])
        except pd.errors.EmptyDataError:
            pmd_df = pd.DataFrame(columns=["chr", "start", "end", "RegionType"])
        if pmd_df.empty:
            continue
        pmd_df["sample_id"] = str(sample_id)
        all_pmds.append(pmd_df)

    if all_pmds:
        all_pmds_df = pd.concat(all_pmds, ignore_index=True)
        all_pmds_df = all_pmds_df[all_pmds_df["chr"].isin(AUTOSOMES)].copy()
    else:
        all_pmds_df = pd.DataFrame(columns=["chr", "start", "end", "RegionType", "sample_id"])

    return all_pmds_df, missing_paths


def merge_pmds_with_sample_support(pmd_df, genome_file):
    if pmd_df.empty:
        return pd.DataFrame(columns=["chr", "start", "end", "sample_ids", "n_samples", "length"])

    merged_bed = pybedtools.BedTool.from_dataframe(pmd_df[["chr", "start", "end", "sample_id"]]).sort(g=genome_file)
    merged_bed = merged_bed.merge(c=4, o="distinct")
    merged_df = merged_bed.to_dataframe(names=["chr", "start", "end", "sample_ids"])
    merged_df["n_samples"] = merged_df["sample_ids"].str.split(",").apply(len)
    merged_df["length"] = merged_df["end"] - merged_df["start"]
    return merged_df


def select_top_pmds_per_cancer_type(train_cancer_samples_df, base_dir, genome_file, top_n_pmd_regions):
    selection_frames = []
    audit_rows = []
    total_missing_pmd_files = 0

    for cancer_type, cancer_samples_df in train_cancer_samples_df.groupby("project_id", sort=True):
        cancer_pmds_df, missing_paths = load_pmd_regions_for_samples(
            cancer_samples_df["sample_id"].astype(str).tolist(),
            base_dir=base_dir,
        )
        total_missing_pmd_files += len(missing_paths)
        merged_df = merge_pmds_with_sample_support(cancer_pmds_df, genome_file=genome_file)
        if not merged_df.empty:
            merged_df = merged_df[(merged_df["length"] >= 1000) & (merged_df["length"] <= 100000000)].copy()

        selected_df = merged_df.sort_values(["n_samples", "length"], ascending=[False, False]).head(top_n_pmd_regions).reset_index(drop=True) if not merged_df.empty else merged_df.copy()
        if not selected_df.empty:
            selected_df["cancer_type"] = str(cancer_type)
            selected_df["support_summary"] = selected_df.apply(lambda row: f"{cancer_type}:{int(row['n_samples'])}", axis=1)
            selected_df["nomination_id"] = [f"{cancer_type}__{idx}" for idx in range(len(selected_df))]
            selection_frames.append(selected_df)

        audit_rows.append({
            "cancer_type": str(cancer_type),
            "n_train_cancer_samples": int(cancer_samples_df["sample_id"].nunique()),
            "n_candidate_regions": int(len(merged_df)),
            "n_selected_regions": int(len(selected_df)),
            "n_missing_pmd_files": int(len(missing_paths)),
        })

    nominated_df = pd.concat(selection_frames, ignore_index=True) if selection_frames else pd.DataFrame(columns=["chr", "start", "end", "sample_ids", "n_samples", "length", "cancer_type", "support_summary", "nomination_id"])
    selection_audit_df = pd.DataFrame(audit_rows)
    return nominated_df, selection_audit_df, total_missing_pmd_files


def merge_nominated_pmds(nominated_df, genome_file):
    if nominated_df.empty:
        return pd.DataFrame(columns=["chr", "start", "end", "cancer_types", "support_summaries", "source_sample_ids", "n_member_pmds", "length", "n_contributing_cancer_types"])

    merge_input_df = nominated_df[["chr", "start", "end", "cancer_type", "support_summary", "sample_ids", "nomination_id"]].copy()
    merged_bed = pybedtools.BedTool.from_dataframe(merge_input_df).sort(g=genome_file)
    merged_bed = merged_bed.merge(c="4,5,6,7", o="distinct,distinct,collapse,count")
    merged_df = merged_bed.to_dataframe(names=["chr", "start", "end", "cancer_types", "support_summaries", "source_sample_ids", "n_member_pmds"])
    merged_df["n_member_pmds"] = merged_df["n_member_pmds"].astype(int)
    merged_df["length"] = merged_df["end"] - merged_df["start"]
    merged_df["n_contributing_cancer_types"] = merged_df["cancer_types"].str.split(",").apply(lambda values: len([value for value in values if value]))
    return merged_df.reset_index(drop=True)


def build_fold_region_sets(train_cancer_samples_df, probe_df, probes_bed, base_dir, genome_file, top_n_pmd_regions):
    nominated_pmds_df, selection_audit_df, total_missing_pmd_files = select_top_pmds_per_cancer_type(
        train_cancer_samples_df=train_cancer_samples_df,
        base_dir=base_dir,
        genome_file=genome_file,
        top_n_pmd_regions=top_n_pmd_regions,
    )
    if nominated_pmds_df.empty:
        raise ValueError("No per-cancer PMD nominations were available for the training tumor samples in this fold.")

    final_pmds_df = merge_nominated_pmds(nominated_pmds_df, genome_file=genome_file)
    if final_pmds_df.empty:
        raise ValueError("No merged PMD regions were available after combining per-cancer nominations in this fold.")

    all_pmds_union_bed = pybedtools.BedTool.from_dataframe(final_pmds_df[["chr", "start", "end"]]).sort(g=genome_file)
    non_region_bed = all_pmds_union_bed.complement(g=genome_file)
    regions_with_probes = non_region_bed.intersect(probes_bed, u=True)
    non_pmd_regions_df = regions_with_probes.to_dataframe(names=["chr", "start", "end"])
    if not non_pmd_regions_df.empty:
        non_pmd_regions_df = non_pmd_regions_df[non_pmd_regions_df["chr"].isin(AUTOSOMES)].copy().reset_index(drop=True)

    if non_pmd_regions_df.empty:
        raise ValueError("No non-PMD intervals with probe support were available in this fold.")

    metadata = {
        "n_train_cancer_samples": int(train_cancer_samples_df["sample_id"].nunique()),
        "n_nomination_cancer_types": int(selection_audit_df.loc[selection_audit_df["n_selected_regions"] > 0, "cancer_type"].nunique()),
        "n_candidate_pmd_regions": int(selection_audit_df["n_candidate_regions"].sum()),
        "n_nominated_pmd_regions": int(len(nominated_pmds_df)),
        "n_top_pmd_regions": int(len(final_pmds_df)),
        "n_final_merged_pmd_regions": int(len(final_pmds_df)),
        "n_non_pmd_regions": int(len(non_pmd_regions_df)),
        "n_missing_pmd_files": int(total_missing_pmd_files),
    }
    return final_pmds_df.reset_index(drop=True), all_pmds_union_bed, non_pmd_regions_df, metadata, selection_audit_df


In [ ]:
def calculate_average_beta_in_region(meth_data, region):
    chrom, start, end = region
    region_meth_data = meth_data.loc[
        (meth_data["CpG_chrm"] == chrom)
        & (meth_data["CpG_beg"] >= start)
        & (meth_data["CpG_end"] <= end)
    ]
    if region_meth_data.empty:
        return None
    return region_meth_data["beta"].mean()


def get_mp_context():
    return mp.get_context("fork") if os.name != "nt" else mp.get_context()


def resolve_worker_count(max_workers, n_tasks):
    if max_workers is None:
        max_workers = os.cpu_count() or 1
    return max(1, min(int(max_workers), int(max(1, n_tasks))))


_FEATURE_EXTRACTION_CONTEXT = {}
_MULTICLASS_FOLD_CONTEXT = {}


def register_mp_callable(func):
    try:
        main_mod = __import__("__main__")
        setattr(main_mod, func.__name__, func)
        func.__module__ = "__main__"
    except Exception:
        pass
    return func


def configure_feature_extraction_context(meth_data_df, region_specs):
    global _FEATURE_EXTRACTION_CONTEXT
    _FEATURE_EXTRACTION_CONTEXT = {
        "meth_data_df": meth_data_df,
        "region_specs": list(region_specs),
    }


def _compute_sample_feature_row(sample_id):
    meth_data_df = _FEATURE_EXTRACTION_CONTEXT["meth_data_df"]
    region_specs = _FEATURE_EXTRACTION_CONTEXT["region_specs"]
    meth_data = meth_data_df.loc[:, ["CpG_chrm", "CpG_beg", "CpG_end", sample_id]].rename(columns={sample_id: "beta"})
    sample_features = {}
    for feature_name, chrom, start, end in region_specs:
        sample_features[feature_name] = calculate_average_beta_in_region(meth_data, (chrom, start, end))
    return sample_id, sample_features


register_mp_callable(_compute_sample_feature_row)


def apply_feature_cap(X):
    if MAX_FEATURES is None:
        return X

    max_features = int(MAX_FEATURES)
    if max_features <= 0:
        raise ValueError("MAX_FEATURES must be a positive integer or None.")
    if X.shape[1] <= max_features:
        return X
    return X.iloc[:, :max_features].copy()


def create_ml_input(regions_df, samples_df, meth_data_df, classification_mode, max_workers=1):
    if regions_df.empty:
        raise ValueError("regions_df is empty, so no features can be created.")

    region_specs = [(row.Index, row.chr, row.start, row.end) for row in regions_df.itertuples()]
    sample_ids = samples_df["sample_id"].astype(str).tolist()
    feature_rows = {}

    worker_count = resolve_worker_count(max_workers=max_workers, n_tasks=len(sample_ids))
    if worker_count > 1 and len(sample_ids) > 1:
        configure_feature_extraction_context(meth_data_df, region_specs)
        with get_mp_context().Pool(processes=worker_count) as pool:
            for sample_id, sample_features in pool.imap_unordered(_compute_sample_feature_row, sample_ids):
                feature_rows[sample_id] = sample_features
    else:
        configure_feature_extraction_context(meth_data_df, region_specs)
        for sample_id in sample_ids:
            returned_sample_id, sample_features = _compute_sample_feature_row(sample_id)
            feature_rows[returned_sample_id] = sample_features

    X = pd.DataFrame.from_dict(feature_rows, orient="index")
    X.index.name = "sample_id"
    X = X.dropna(axis=1, how="all")
    if X.shape[1] == 0:
        raise ValueError("No usable region features were created. All candidate regions were missing CpGs across all samples.")
    X = apply_feature_cap(X)
    if X.shape[1] == 0:
        raise ValueError("MAX_FEATURES removed all usable region features. Increase MAX_FEATURES or set it to None.")

    y = build_multiclass_labels(samples_df, classification_mode).loc[X.index]
    return X, y


def prepare_train_test_data(X, y, train_sample_ids, test_sample_ids):
    if X.empty or X.shape[1] == 0:
        raise ValueError("X has no feature columns to train on.")

    train_sample_ids = [sample_id for sample_id in train_sample_ids if sample_id in X.index]
    test_sample_ids = [sample_id for sample_id in test_sample_ids if sample_id in X.index]

    X_train = X.loc[train_sample_ids]
    X_test = X.loc[test_sample_ids]
    y_train = y.loc[train_sample_ids]
    y_test = y.loc[test_sample_ids]

    X_train = X_train.dropna(axis=1, how="all")
    if X_train.shape[1] == 0:
        raise ValueError("All training feature columns are entirely missing after filtering, so classification cannot run.")

    X_test = X_test.loc[:, X_train.columns]
    imputer = SimpleImputer(strategy="mean")
    X_train = pd.DataFrame(imputer.fit_transform(X_train), index=X_train.index, columns=X_train.columns)
    X_test = pd.DataFrame(imputer.transform(X_test), index=X_test.index, columns=X_test.columns)
    return X_train, X_test, y_train, y_test


def compute_macro_negative_metrics(y_true, y_pred, classes):
    mcm = multilabel_confusion_matrix(y_true, y_pred, labels=classes)
    specificities = []
    negative_precisions = []
    for cm in mcm:
        tn, fp, fn, tp = cm.ravel()
        specificities.append(tn / (tn + fp) if (tn + fp) else 0.0)
        negative_precisions.append(tn / (tn + fn) if (tn + fn) else 0.0)
    return float(np.mean(specificities)), float(np.mean(negative_precisions)), specificities, negative_precisions


def summarize_multiclass_classification(y_true, y_pred, y_score_df, analysis, n_regions, n_shortened=0, mean_length_ratio=np.nan):
    classes = y_score_df.columns.astype(str).tolist()
    _, _, macro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    macro_specificity, macro_negative_precision, _, _ = compute_macro_negative_metrics(y_true, y_pred, classes)
    return {
        "analysis": analysis,
        "n_regions": int(n_regions),
        "n_shortened": int(n_shortened),
        "mean_length_ratio": float(mean_length_ratio) if not pd.isna(mean_length_ratio) else np.nan,
        "mcc": float(matthews_corrcoef(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(macro_f1),
        "macro_specificity": macro_specificity,
        "macro_negative_precision": macro_negative_precision,
    }


def build_per_class_metrics(y_true, y_pred, classes, analysis):
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=classes,
        zero_division=0,
    )
    _, _, class_specificity, class_negative_precision = compute_macro_negative_metrics(y_true, y_pred, classes)
    return pd.DataFrame({
        "analysis": analysis,
        "class_label": classes,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "specificity": class_specificity,
        "negative_precision": class_negative_precision,
        "support": support,
    })


def build_confusion_records(y_true, y_pred, classes, analysis):
    cm = confusion_matrix(y_true, y_pred, labels=classes)
    records = []
    for i, true_label in enumerate(classes):
        for j, predicted_label in enumerate(classes):
            records.append({
                "analysis": analysis,
                "true_label": true_label,
                "predicted_label": predicted_label,
                "count": int(cm[i, j]),
            })
    return pd.DataFrame(records)


def run_ml_classification(X, y, train_sample_ids, test_sample_ids, analysis, class_weight="balanced", region_metadata=None):
    X_train, X_test, y_train, y_test = prepare_train_test_data(X, y, train_sample_ids, test_sample_ids)
    rf = RandomForestClassifier(n_estimators=500, random_state=42, class_weight=class_weight)
    rf.fit(X_train, y_train)
    y_pred = pd.Series(rf.predict(X_test), index=y_test.index)
    classes = [str(c) for c in rf.classes_]
    y_score_df = pd.DataFrame(rf.predict_proba(X_test), index=y_test.index, columns=classes)
    region_metadata = region_metadata if region_metadata is not None else pd.DataFrame()
    metrics = summarize_multiclass_classification(
        y_true=y_test,
        y_pred=y_pred,
        y_score_df=y_score_df,
        analysis=analysis,
        n_regions=X_train.shape[1],
        n_shortened=region_metadata.get("match_type", pd.Series(dtype=object)).eq("shortened_interval").sum() if not region_metadata.empty else 0,
        mean_length_ratio=region_metadata.get("length_ratio", pd.Series(dtype=float)).mean() if not region_metadata.empty else np.nan,
    )
    class_metrics_df = build_per_class_metrics(y_test, y_pred, classes, analysis)
    confusion_df = build_confusion_records(y_test, y_pred, classes, analysis)
    return metrics, class_metrics_df, confusion_df


def run_majority_class_baseline(y, train_sample_ids, test_sample_ids, analysis="Majority class"):
    X_train, X_test, y_train, y_test = prepare_train_test_data(
        X=pd.DataFrame(index=y.index, data={"dummy": np.zeros(len(y))}),
        y=y,
        train_sample_ids=train_sample_ids,
        test_sample_ids=test_sample_ids,
    )
    del X_train, X_test
    classes = sorted(pd.unique(y.astype(str)).tolist())
    majority_label = y_train.astype(str).value_counts().idxmax()
    y_pred = pd.Series(majority_label, index=y_test.index)
    y_score_df = pd.DataFrame(0.0, index=y_test.index, columns=classes)
    y_score_df.loc[:, majority_label] = 1.0
    metrics = summarize_multiclass_classification(
        y_true=y_test,
        y_pred=y_pred,
        y_score_df=y_score_df,
        analysis=analysis,
        n_regions=0,
    )
    class_metrics_df = build_per_class_metrics(y_test, y_pred, classes, analysis)
    confusion_df = build_confusion_records(y_test, y_pred, classes, analysis)
    return metrics, class_metrics_df, confusion_df


def summarize_fold_results(fold_results_df, metadata):
    rows = []
    for analysis, analysis_df in fold_results_df.groupby("analysis", sort=False):
        row = {
            "classification_mode": metadata["classification_mode"],
            "cohort_label": metadata["cohort_label"],
            "analysis": analysis,
            "n_samples": metadata["n_samples"],
            "n_tumors": metadata["n_tumors"],
            "n_normals": metadata["n_normals"],
            "n_projects": metadata["n_projects"],
            "n_classes": metadata["n_classes"],
            "n_folds_completed": int(analysis_df["fold"].nunique()),
            "mean_n_regions": analysis_df["n_regions"].mean(),
            "mean_n_shortened": analysis_df["n_shortened"].mean(),
            "mean_length_ratio": analysis_df["mean_length_ratio"].mean(),
        }
        for metric in METRIC_COLS:
            row[f"{metric}_mean"] = analysis_df[metric].mean()
            row[f"{metric}_std"] = analysis_df[metric].std()
            row[f"{metric}_min"] = analysis_df[metric].min()
            row[f"{metric}_median"] = analysis_df[metric].median()
            row[f"{metric}_max"] = analysis_df[metric].max()
        rows.append(row)
    return pd.DataFrame(rows)


def summarize_per_class_results(class_results_df, metadata):
    rows = []
    for (analysis, class_label), class_df in class_results_df.groupby(["analysis", "class_label"], sort=False):
        rows.append({
            "classification_mode": metadata["classification_mode"],
            "cohort_label": metadata["cohort_label"],
            "analysis": analysis,
            "class_label": class_label,
            "precision_mean": class_df["precision"].mean(),
            "recall_mean": class_df["recall"].mean(),
            "f1_mean": class_df["f1"].mean(),
            "specificity_mean": class_df["specificity"].mean(),
            "negative_precision_mean": class_df["negative_precision"].mean(),
            "support_total": class_df["support"].sum(),
            "n_folds_completed": int(class_df["fold"].nunique()),
        })
    return pd.DataFrame(rows)


def plot_multiclass_analysis_results(fold_results_df, cohort_label):
    metrics_to_plot = ["mcc", "balanced_accuracy", "macro_f1", "macro_specificity", "macro_negative_precision"]
    fig, axes = plt.subplots(1, len(metrics_to_plot), figsize=(24, 5))

    for ax, metric in zip(axes, metrics_to_plot):
        pmd_values = fold_results_df.loc[fold_results_df["analysis"] == "PMD", metric].dropna()
        random_values = fold_results_df.loc[fold_results_df["analysis"] == "Random", metric].dropna()
        baseline_values = fold_results_df.loc[fold_results_df["analysis"] == "Majority class", metric].dropna()

        plot_data = []
        plot_positions = []
        if not pmd_values.empty:
            plot_data.append(pmd_values)
            plot_positions.append(1)
        if not random_values.empty:
            plot_data.append(random_values)
            plot_positions.append(2)

        if plot_data:
            ax.boxplot(
                plot_data,
                positions=plot_positions,
                widths=0.55,
                patch_artist=True,
                boxprops={"facecolor": "lightgray", "edgecolor": "dimgray", "linewidth": 1.5},
                medianprops={"color": "black", "linewidth": 2},
                whiskerprops={"color": "dimgray", "linewidth": 1.5},
                capprops={"color": "dimgray", "linewidth": 1.5},
                flierprops={"marker": "o", "markerfacecolor": "gray", "markeredgecolor": "dimgray", "alpha": 0.6, "markersize": 4},
            )

        if not baseline_values.empty:
            ax.scatter(
                np.full(len(baseline_values), 3.0),
                baseline_values,
                color="firebrick",
                s=45,
                alpha=0.8,
                label="Majority class folds" if metric == metrics_to_plot[0] else None,
            )
            ax.scatter(
                [3.0],
                [baseline_values.median()],
                color="darkred",
                marker="D",
                s=80,
                label="Majority class median" if metric == metrics_to_plot[0] else None,
            )

        ax.set_xticks([1, 2, 3])
        ax.set_xticklabels(["PMD boxplot", "Random boxplot", "Majority baseline"])
        ax.set_title(metric.replace("_", " ").title())
        ax.set_ylabel("Score")
        ax.grid(axis="y", alpha=0.25)
        ax.set_axisbelow(True)

    handles, labels = axes[0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, loc="upper center", ncol=2, frameon=False)
    fig.suptitle(f"Fold-level multiclass CV scores: {cohort_label}", y=1.06)
    fig.tight_layout(rect=[0, 0, 1, 0.93])
    plt.show()


def plot_aggregated_confusion_matrix(confusion_results_df, analysis, cohort_label):
    analysis_df = confusion_results_df.loc[confusion_results_df["analysis"] == analysis].copy()
    if analysis_df.empty:
        print(f"No confusion-matrix rows are available for analysis={analysis!r}.")
        return

    pivot = analysis_df.pivot_table(index="true_label", columns="predicted_label", values="count", aggfunc="sum", fill_value=0)
    pivot = pivot.sort_index().sort_index(axis=1)
    normalized = pivot.div(pivot.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)

    fig, ax = plt.subplots(figsize=(1.2 * len(normalized.columns) + 2, 1.0 * len(normalized.index) + 2))
    im = ax.imshow(normalized.to_numpy(), cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(normalized.columns)))
    ax.set_xticklabels(normalized.columns, rotation=90)
    ax.set_yticks(range(len(normalized.index)))
    ax.set_yticklabels(normalized.index)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_title(f"Aggregated normalized confusion matrix: {analysis}\n{cohort_label}")

    for i in range(normalized.shape[0]):
        for j in range(normalized.shape[1]):
            value = normalized.iat[i, j]
            ax.text(j, i, f"{value:.2f}", ha="center", va="center", color="black" if value < 0.65 else "white", fontsize=9)

    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    plt.show()


def build_multiclass_conclusion(summary_df):
    pmd_row = summary_df.loc[summary_df["analysis"] == "PMD"]
    random_row = summary_df.loc[summary_df["analysis"] == "Random"]
    if pmd_row.empty or random_row.empty:
        return "Evidence is inconclusive because one or more comparison analyses did not produce summary rows."

    pmd_mcc = float(pmd_row.iloc[0]["mcc_mean"])
    pmd_bal = float(pmd_row.iloc[0]["balanced_accuracy_mean"])
    rand_mcc = float(random_row.iloc[0]["mcc_mean"])
    rand_bal = float(random_row.iloc[0]["balanced_accuracy_mean"])
    if pmd_mcc > rand_mcc and pmd_bal > rand_bal:
        return "PMD regions outperform the PMD-free random baseline on mean MCC and mean balanced accuracy across folds."
    return "Evidence is weak or inconclusive: PMD regions do not clearly outperform the PMD-free random baseline on the main multiclass summary metrics."


def configure_multiclass_fold_context(context):
    global _MULTICLASS_FOLD_CONTEXT
    _MULTICLASS_FOLD_CONTEXT = context


def _run_multiclass_fold_task(task):
    context = _MULTICLASS_FOLD_CONTEXT
    fold_idx = int(task["fold"])
    train_idx = task["train_idx"]
    test_idx = task["test_idx"]
    analysis_samples_df = context["analysis_samples_df"]
    cohort_label = context["cohort_label"]
    classification_mode = context["classification_mode"]
    feature_workers = context["feature_workers"]

    train_df = analysis_samples_df.iloc[train_idx].reset_index(drop=True)
    test_df = analysis_samples_df.iloc[test_idx].reset_index(drop=True)
    train_sample_ids = train_df["sample_id"].astype(str).tolist()
    test_sample_ids = test_df["sample_id"].astype(str).tolist()
    train_cancer_samples_df = train_df[train_df["sample_type"] == "Primary Tumor"].copy()

    warnings = []
    fold_results = []
    class_frames = []
    confusion_frames = []
    random_audit_frames = []

    try:
        final_pmds_df, all_pmds_union_bed, non_pmd_regions_df, region_meta, selection_audit_df = build_fold_region_sets(
            train_cancer_samples_df=train_cancer_samples_df,
            probe_df=context["probe_df"],
            probes_bed=context["probes_bed"],
            base_dir=context["base_dir"],
            genome_file=context["genome_file"],
            top_n_pmd_regions=context["top_n_pmd_regions"],
        )
    except ValueError as exc:
        warnings.append({"fold": fold_idx, "warning": str(exc)})
        return {
            "fold": fold_idx,
            "fold_results": fold_results,
            "class_df": pd.DataFrame(),
            "confusion_df": pd.DataFrame(),
            "random_audit_df": pd.DataFrame(),
            "selection_audit_df": pd.DataFrame(),
            "warnings": warnings,
        }

    selection_audit_df = selection_audit_df.copy()
    selection_audit_df["fold"] = fold_idx
    selection_audit_df["cohort_label"] = cohort_label
    selection_audit_df["classification_mode"] = classification_mode
    selection_audit_df["n_final_merged_regions"] = int(region_meta["n_final_merged_pmd_regions"])

    X_pmd, y_multiclass = create_ml_input(
        final_pmds_df,
        analysis_samples_df,
        context["meth_data_df"],
        classification_mode,
        max_workers=feature_workers,
    )
    pmd_metrics, pmd_class_df, pmd_confusion_df = run_ml_classification(
        X_pmd,
        y_multiclass,
        train_sample_ids,
        test_sample_ids,
        analysis="PMD",
    )
    pmd_metrics.update({
        "fold": fold_idx,
        "classification_mode": classification_mode,
        "cohort_label": cohort_label,
        "n_train_samples": len(train_df),
        "n_test_samples": len(test_df),
        "n_train_tumors": int((train_df["sample_type"] == "Primary Tumor").sum()),
        "n_train_normals": int((train_df["sample_type"] == "Solid Tissue Normal").sum()),
    })
    pmd_metrics.update(region_meta)
    fold_results.append(pmd_metrics)
    pmd_class_df["fold"] = fold_idx
    pmd_confusion_df["fold"] = fold_idx
    class_frames.append(pmd_class_df)
    confusion_frames.append(pmd_confusion_df)

    baseline_metrics, baseline_class_df, baseline_confusion_df = run_majority_class_baseline(
        y_multiclass,
        train_sample_ids,
        test_sample_ids,
    )
    baseline_metrics.update({
        "fold": fold_idx,
        "classification_mode": classification_mode,
        "cohort_label": cohort_label,
        "n_train_samples": len(train_df),
        "n_test_samples": len(test_df),
        "n_train_tumors": int((train_df["sample_type"] == "Primary Tumor").sum()),
        "n_train_normals": int((train_df["sample_type"] == "Solid Tissue Normal").sum()),
    })
    baseline_metrics.update(region_meta)
    fold_results.append(baseline_metrics)
    baseline_class_df["fold"] = fold_idx
    baseline_confusion_df["fold"] = fold_idx
    class_frames.append(baseline_class_df)
    confusion_frames.append(baseline_confusion_df)

    random_regions_df, skipped_df, _ = get_random_regions(
        pmd_regions_df=final_pmds_df,
        non_pmd_regions_df=non_pmd_regions_df,
        meth_data_df=context["meth_data_df"],
        seed=context["random_region_seed"] + fold_idx,
    )

    if random_regions_df.empty:
        warnings.append({
            "fold": fold_idx,
            "warning": "No PMD-free random regions could be generated for this fold.",
        })
    else:
        validation_df = validate_random_regions(
            random_regions_df=random_regions_df,
            non_pmd_regions_df=non_pmd_regions_df,
            pmd_union_bed=all_pmds_union_bed,
            genome_file=context["genome_file"],
        )
        validation_df["fold"] = fold_idx
        random_audit_frames.append(validation_df)

        X_random, y_random = create_ml_input(
            random_regions_df,
            analysis_samples_df,
            context["meth_data_df"],
            classification_mode,
            max_workers=feature_workers,
        )
        random_metrics, random_class_df, random_confusion_df = run_ml_classification(
            X_random,
            y_random,
            train_sample_ids,
            test_sample_ids,
            analysis="Random",
            region_metadata=random_regions_df,
        )
        random_metrics.update({
            "fold": fold_idx,
            "classification_mode": classification_mode,
            "cohort_label": cohort_label,
            "n_train_samples": len(train_df),
            "n_test_samples": len(test_df),
            "n_train_tumors": int((train_df["sample_type"] == "Primary Tumor").sum()),
            "n_train_normals": int((train_df["sample_type"] == "Solid Tissue Normal").sum()),
            "n_random_skipped": int(len(skipped_df)),
        })
        random_metrics.update(region_meta)
        fold_results.append(random_metrics)
        random_class_df["fold"] = fold_idx
        random_confusion_df["fold"] = fold_idx
        class_frames.append(random_class_df)
        confusion_frames.append(random_confusion_df)

        if (random_regions_df["match_type"] == "shortened_interval").any():
            warnings.append({
                "fold": fold_idx,
                "warning": "Some random regions were shortened to stay PMD-free within a single complement interval.",
            })

    if not random_regions_df.empty and len(random_regions_df) < len(final_pmds_df):
        warnings.append({
            "fold": fold_idx,
            "warning": "Fewer PMD-free random regions were available than PMD regions in this fold.",
        })

    class_df = pd.concat(class_frames, ignore_index=True) if class_frames else pd.DataFrame()
    confusion_df = pd.concat(confusion_frames, ignore_index=True) if confusion_frames else pd.DataFrame()
    random_audit_df = pd.concat(random_audit_frames, ignore_index=True) if random_audit_frames else pd.DataFrame()
    return {
        "fold": fold_idx,
        "fold_results": fold_results,
        "class_df": class_df,
        "confusion_df": confusion_df,
        "random_audit_df": random_audit_df,
        "selection_audit_df": selection_audit_df,
        "warnings": warnings,
    }


register_mp_callable(_run_multiclass_fold_task)


def run_multiclass_analysis(analysis_samples_df, cohort_label, classification_mode, selected_tumor_types, cv_n_splits, meth_data_df, probe_df, probes_bed, base_dir, genome_file, top_n_pmd_regions, random_region_seed, parallelize_folds=True, max_fold_workers=1, max_feature_workers=1):
    labels = build_multiclass_labels(analysis_samples_df, classification_mode)
    splitter = StratifiedKFold(n_splits=cv_n_splits, shuffle=True, random_state=42)
    split_tasks = [
        {"fold": fold_idx, "train_idx": train_idx, "test_idx": test_idx}
        for fold_idx, (train_idx, test_idx) in enumerate(splitter.split(analysis_samples_df, labels), start=1)
    ]

    fold_worker_count = resolve_worker_count(max_workers=max_fold_workers, n_tasks=len(split_tasks))
    use_parallel_folds = bool(parallelize_folds and fold_worker_count > 1 and len(split_tasks) > 1)
    feature_worker_count = 1 if use_parallel_folds else resolve_worker_count(max_workers=max_feature_workers, n_tasks=len(analysis_samples_df))

    fold_context = {
        "analysis_samples_df": analysis_samples_df,
        "cohort_label": cohort_label,
        "classification_mode": classification_mode,
        "meth_data_df": meth_data_df,
        "probe_df": probe_df,
        "probes_bed": probes_bed,
        "base_dir": base_dir,
        "genome_file": genome_file,
        "top_n_pmd_regions": top_n_pmd_regions,
        "random_region_seed": random_region_seed,
        "feature_workers": feature_worker_count,
    }

    configure_multiclass_fold_context(fold_context)
    fold_outputs = []
    if use_parallel_folds:
        with get_mp_context().Pool(processes=fold_worker_count) as pool:
            for fold_output in tqdm(pool.imap_unordered(_run_multiclass_fold_task, split_tasks), total=len(split_tasks), desc=f"{cohort_label} folds"):
                fold_outputs.append(fold_output)
    else:
        for task in tqdm(split_tasks, total=len(split_tasks), desc=f"{cohort_label} folds"):
            fold_outputs.append(_run_multiclass_fold_task(task))

    fold_outputs = sorted(fold_outputs, key=lambda item: item["fold"])
    fold_results = []
    class_frames = []
    confusion_frames = []
    random_audit_frames = []
    selection_audit_frames = []
    warnings = []
    for fold_output in fold_outputs:
        fold_results.extend(fold_output["fold_results"])
        if not fold_output["class_df"].empty:
            class_frames.append(fold_output["class_df"])
        if not fold_output["confusion_df"].empty:
            confusion_frames.append(fold_output["confusion_df"])
        if not fold_output["random_audit_df"].empty:
            random_audit_frames.append(fold_output["random_audit_df"])
        if not fold_output["selection_audit_df"].empty:
            selection_audit_frames.append(fold_output["selection_audit_df"])
        warnings.extend(fold_output["warnings"])

    fold_results_df = pd.DataFrame(fold_results)
    if fold_results_df.empty:
        raise ValueError(f"No fold-level results could be generated for cohort {cohort_label}.")

    metadata = {
        "classification_mode": classification_mode,
        "cohort_label": cohort_label,
        "n_samples": int(analysis_samples_df["sample_id"].nunique()),
        "n_tumors": int((analysis_samples_df["sample_type"] == "Primary Tumor").sum()),
        "n_normals": int((analysis_samples_df["sample_type"] == "Solid Tissue Normal").sum()),
        "n_projects": int(analysis_samples_df["project_id"].nunique()),
        "n_classes": int(labels.nunique()),
        "n_folds": cv_n_splits,
        "selected_tumor_types": list(selected_tumor_types),
        "runtime_parallelize_folds": bool(use_parallel_folds),
        "runtime_max_fold_workers": int(fold_worker_count),
        "runtime_max_feature_workers": int(feature_worker_count),
        "runtime_max_features": None if MAX_FEATURES is None else int(MAX_FEATURES),
    }
    random_audit_df = pd.concat(random_audit_frames, ignore_index=True) if random_audit_frames else pd.DataFrame()
    selection_audit_df = pd.concat(selection_audit_frames, ignore_index=True) if selection_audit_frames else pd.DataFrame()
    class_results_df = pd.concat(class_frames, ignore_index=True) if class_frames else pd.DataFrame()
    confusion_results_df = pd.concat(confusion_frames, ignore_index=True) if confusion_frames else pd.DataFrame()
    summary_df = summarize_fold_results(fold_results_df, metadata)
    per_class_summary_df = summarize_per_class_results(class_results_df, metadata) if not class_results_df.empty else pd.DataFrame()
    return fold_results_df, class_results_df, confusion_results_df, random_audit_df, selection_audit_df, summary_df, per_class_summary_df, warnings, metadata



In [ ]:
(
    analysis_fold_results_df,
    analysis_class_results_df,
    analysis_confusion_results_df,
    analysis_random_audit_df,
    analysis_region_selection_audit_df,
    analysis_cohort_summary_df,
    analysis_per_class_summary_df,
    analysis_warnings,
    analysis_metadata,
) = run_multiclass_analysis(
    analysis_samples_df=analysis_samples_df,
    cohort_label=analysis_label,
    classification_mode=classification_mode,
    selected_tumor_types=selected_tumor_types,
    cv_n_splits=cv_n_splits,
    meth_data_df=meth_data_df,
    probe_df=probe_df,
    probes_bed=probes_bed,
    base_dir=base_dir,
    genome_file=genome_file,
    top_n_pmd_regions=top_n_pmd_regions,
    random_region_seed=random_region_seed,
    parallelize_folds=parallelize_folds,
    max_fold_workers=max_fold_workers,
    max_feature_workers=max_feature_workers,
)


In [ ]:
plot_multiclass_analysis_results(analysis_fold_results_df, analysis_label)
plot_aggregated_confusion_matrix(analysis_confusion_results_df, analysis="PMD", cohort_label=analysis_label)
analysis_cohort_summary_df, analysis_per_class_summary_df.head(30), analysis_region_selection_audit_df.head(30), analysis_random_audit_df.head(20), pd.DataFrame([analysis_metadata]), pd.DataFrame(analysis_warnings), build_multiclass_conclusion(analysis_cohort_summary_df)


In [ ]:
plot_aggregated_confusion_matrix(analysis_confusion_results_df, analysis="Random", cohort_label=analysis_label)

In [ ]:
analysis_fold_results_df[["fold", "analysis", "n_regions"]]